In [1]:
import torch

x = torch.ones(5) # input tensor
y = torch.zeros(3) # expected tensor
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
print(loss)

tensor(1.4497, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


In [2]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x10ee5f6d0>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x10f6d11b0>


In [ ]:
# Compute gradient: dloss/dw, dloss/db
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.2449, 0.1446, 0.3046],
        [0.2449, 0.1446, 0.3046],
        [0.2449, 0.1446, 0.3046],
        [0.2449, 0.1446, 0.3046],
        [0.2449, 0.1446, 0.3046]])
tensor([0.2449, 0.1446, 0.3046])


In [6]:
# Disable gradient tracking 
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

# Alternative
z = torch.matmul(x, w) + b
z_det = z.detach()
print(z_det.requires_grad)

# Reasons to disable gradient tracking
# * To make some parameters in nn as frozen parameters
# * To speed up computations when only doing forward pass

True
False
False


In [ ]:
# Jacobian Products
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t() # add 1 on each element, power by 2, then transpose (shape (5,4))
# there are 20 elements in out, we need to give a weight to each element:
# how much does each element count to the derivative, and here we give each element a 1
# this is equal to: out.sum().backward(retain_graph=True)
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First Call\n{inp.grad}")
# Gradients are aggregated
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond Call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

# retain_graph=True keeps the intermediate data needed to run backward again 
# through the same computation graph.
# By default, PyTorch frees this data after backward() to save memory
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()  # x.grad = 4
# y.backward()  # Error: required intermediate data was freed

First Call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second Call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
